In [1]:
library(microDecon)
library(tidyverse)
library(phyloseq)
library(readxl)

── Attaching core tidyverse packages ──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.2.1     ✔ readr     2.2.0
✔ forcats   1.0.1     ✔ stringr   1.6.0
✔ ggplot2   4.0.3     ✔ tibble    3.3.1
✔ lubridate 1.9.5     ✔ tidyr     1.3.2
✔ purrr     1.2.2     
── Conflicts ────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors


In [2]:
extraction_batches_raw <- readxl::read_xlsx("Extraction Batches.xlsx")

New names:
• `` -> `...5`


In [89]:
extraction_batches <- dplyr::mutate(extraction_batches_raw,
                              sequenced_tag = if_else(lab_blank %in% Not_sequenced, "Not_sequenced", "Sequenced"),
                              lab_blank = if_else(lab_blank %in% Not_sequenced, NA_character_, lab_blank),
                              date = as.Date(date, format = "%d/%m/%Y"),
                              #Set date as NA if thata is not sequenced to ensure correct batch assignment
                              assigned_date = if_else(sequenced_tag == "Not_sequenced", NA, date)) |>
                              #This assigns 
                              tidyr::fill(assigned_date, lab_blank) |>
                       dplyr::group_by(assigned_date, lab_blank, field_blank) |>
                       dplyr::mutate(batch = paste0("Batch_", dplyr::cur_group_id())) |>
                       dplyr::ungroup() |>
                       dplyr::mutate(batch = factor(batch, levels = sort(unique(batch)))) |>
                       dplyr::rename(field_sample = sample_id) |>
                       dplyr::select(batch, date, assigned_date, field_sample, lab_blank, field_blank, sequenced_tag)

In [90]:
sample_n(extraction_batches, 5)

batch,date,assigned_date,field_sample,lab_blank,field_blank,sequenced_tag
<fct>,<date>,<date>,<chr>,<chr>,<chr>,<chr>
Batch_6,2026-05-12,2026-05-12,T1SS3,T1SSL1,NA,Sequenced
Batch_10,2026-05-14,2026-05-14,T3W2,T3L3,T3WB,Sequenced
Batch_7,2026-05-12,2026-05-12,T3IS2,T3ISL3,NA,Sequenced
Batch_2,2026-05-07,2026-05-07,T1SG3,T1SGL1,NA,Sequenced
Batch_3,2026-05-08,2026-05-08,T1IS3,T1ISL1,NA,Sequenced


In [100]:
extraction_batches_all <- tidyr::pivot_longer(extraction_batches, cols = c(field_sample, lab_blank, field_blank), names_to = "source", values_to = "sample_id") |>
                           dplyr::filter(!is.na(sample_id)) |>
                           dplyr::distinct(sample_id, source, .keep_all = TRUE)
sample_n(extraction_batches_all, size = 5)
dim(extraction_batches_all)

batch,date,assigned_date,sequenced_tag,source,sample_id
<fct>,<date>,<date>,<chr>,<chr>,<chr>
Batch_2,2026-05-07,2026-05-07,Sequenced,field_sample,T1SG1
Batch_3,2026-05-09,2026-05-08,Not_sequenced,field_sample,T2IS1
Batch_9,2026-05-13,2026-05-13,Sequenced,field_sample,T3SS3
Batch_5,2026-05-09,2026-05-09,Sequenced,field_sample,T2W1
Batch_4,2026-05-08,2026-05-08,Sequenced,lab_blank,T3SGL3


[1] 53  6

In [101]:
(phyloseq_16S <- read_rds("16S_phyloseq.rds"))

phyloseq-class experiment-level object
otu_table()   OTU Table:         [ 17269 taxa and 53 samples ]
sample_data() Sample Data:       [ 53 samples by 7 sample variables ]
tax_table()   Taxonomy Table:    [ 17269 taxa by 7 taxonomic ranks ]
phy_tree()    Phylogenetic Tree: [ 17269 tips and 17253 internal nodes ]

In [108]:
temp_sample_data <- data.frame(sample_data(phyloseq_16S))
temp_sample_data$Read_tag <- row.names(temp_sample_data)
dim(temp_sample_data)

[1] 53  8

In [109]:
head(temp_sample_data, 3)

,transect,habitat,microbial_community,sample,plant_species_1,plant_species_2,blank_group,Read_tag
,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>
RemovePrimer_Final.T1IS1_8522605005578,T1,intertidal_saltmarsh,prokaryote,T1IS1,Salicornia_tegataria,Triglochin_striata,Real Sample,RemovePrimer_Final.T1IS1_8522605005578
RemovePrimer_Final.T1IS2_8522605005579,T1,intertidal_saltmarsh,prokaryote,T1IS2,Salicornia_tegataria,Triglochin_striata,Real Sample,RemovePrimer_Final.T1IS2_8522605005579
RemovePrimer_Final.T1IS3_8522605005580,T1,intertidal_saltmarsh,prokaryote,T1IS3,Salicornia_tegataria,Triglochin_striata,Real Sample,RemovePrimer_Final.T1IS3_8522605005580


In [110]:
(extraction_batches_all <- merge(temp_sample_data, extraction_batches, by.x = "sample", by.y = "field_sample")) |>  dim()

[1] 39 14

In [111]:
head(extraction_batches_all, 3)

,sample,transect,habitat,microbial_community,plant_species_1,plant_species_2,blank_group,Read_tag,batch,date,assigned_date,lab_blank,field_blank,sequenced_tag
,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<fct>,<date>,<date>,<chr>,<chr>,<chr>
1,T1IS1,T1,intertidal_saltmarsh,prokaryote,Salicornia_tegataria,Triglochin_striata,Real Sample,RemovePrimer_Final.T1IS1_8522605005578,Batch_3,2026-05-08,2026-05-08,T1ISL1,NA,Sequenced
2,T1IS2,T1,intertidal_saltmarsh,prokaryote,Salicornia_tegataria,Triglochin_striata,Real Sample,RemovePrimer_Final.T1IS2_8522605005579,Batch_3,2026-05-08,2026-05-08,T1ISL1,NA,Sequenced
3,T1IS3,T1,intertidal_saltmarsh,prokaryote,Salicornia_tegataria,Triglochin_striata,Real Sample,RemovePrimer_Final.T1IS3_8522605005580,Batch_3,2026-05-08,2026-05-08,T1ISL1,NA,Sequenced


In [32]:
# tibble::rownames_to_column(var = "Read_tag") |>
# data.frame() |>
# dplyr::mutate(tmp_sample = sample) |>
# tibble::column_to_rownames(var = "tmp_sample")     

In [33]:
#head(temp_sample_data)

In [34]:
#head(extraction_batches)

In [35]:
#sample_data(phyloseq_16S) <- temp_sample_data

In [36]:
#head(temp_sample_data)

In [43]:
extraction_batches_all$lab_blank

[1] "T1ISL1" "T1ISL1" "T1ISL1" "T1SGL1" "T1SGL1" "T1SGL1" "TSGBL1" "T1SSL1"
 [9] "T1SSL1" "T1SSL1" "T1L1"   "T1L1"   "T1L1"   "T1ISL1" "T1ISL1" "T1ISL1"
[17] "T1SGL1" "T1SGL1" "T1SGL1" "TSGBL1" "T2SSL2" "T2SSL2" "T2SSL2" "T2L2"  
[25] "T2L2"   "T2L2"   "T3ISL3" "T3ISL3" "T3ISL3" "T3SGL3" "T3SGL3" "T3SGL3"
[33] "TSGBL1" "T3SSL3" "T3SSL3" "T3SSL3" "T3L3"   "T3L3"   "T3L3"

In [51]:
extraction_batch <- extraction_batches_all |> dplyr::filter(batch == "Batch_1")

In [52]:
extraction_batch

sample,transect,habitat,microbial_community,plant_species_1,plant_species_2,blank_group,Read_tag,batch,date,assigned_date,lab_blank,field_blank,sequenced_tag
<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<fct>,<date>,<date>,<chr>,<chr>,<chr>
T1W1,T1,water,prokaryote,NA,NA,Real Sample,RemovePrimer_Final.T1W1_8522605005554,Batch_1,2026-05-07,2026-05-07,T1L1,T1WB,Sequenced
T1W2,T1,water,prokaryote,NA,NA,Real Sample,RemovePrimer_Final.T1W2_8522605005555,Batch_1,2026-05-07,2026-05-07,T1L1,T1WB,Sequenced
T1W3,T1,water,prokaryote,NA,NA,Real Sample,RemovePrimer_Final.T1W3_8522605005556,Batch_1,2026-05-07,2026-05-07,T1L1,T1WB,Sequenced


In [48]:
lapply(unique(extraction_batches_all$batch), function(current_batch) {
    
      extraction_batch <- extraction_batches_all |> dplyr::filter(batch == current_batch)

      #print(extraction_batch$ )
          #physeq_subset <- subset_samples(physeq, sample_id %in% samples_to_keep)
        
})

sample,transect,habitat,microbial_community,plant_species_1,plant_species_2,blank_group,Read_tag,batch,date,assigned_date,lab_blank,field_blank,sequenced_tag
<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<fct>,<date>,<date>,<chr>,<chr>,<chr>
T1IS1,T1,intertidal_saltmarsh,prokaryote,Salicornia_tegataria,Triglochin_striata,Real Sample,RemovePrimer_Final.T1IS1_8522605005578,Batch_3,2026-05-08,2026-05-08,T1ISL1,NA,Sequenced
T1IS2,T1,intertidal_saltmarsh,prokaryote,Salicornia_tegataria,Triglochin_striata,Real Sample,RemovePrimer_Final.T1IS2_8522605005579,Batch_3,2026-05-08,2026-05-08,T1ISL1,NA,Sequenced
T1IS3,T1,intertidal_saltmarsh,prokaryote,Salicornia_tegataria,Triglochin_striata,Real Sample,RemovePrimer_Final.T1IS3_8522605005580,Batch_3,2026-05-08,2026-05-08,T1ISL1,NA,Sequenced
sample,transect,habitat,microbial_community,plant_species_1,plant_species_2,blank_group,Read_tag,batch,date,assigned_date,lab_blank,field_blank,sequenced_tag
<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<fct>,<date>,<date>,<chr>,<chr>,<chr>
T1SG1,T1,seagrass,prokaryote,zostera_capensis,zostera_capensis,Real Sample,RemovePrimer_Final.T1SG1_8522605005566,Batch_2,2026-05-07,2026-05-07,T1SGL1,NA,Sequenced
T1SG2,T1,seagrass,prokaryote,zostera_capensis,zostera_capensis,Real Sample,RemovePrimer_Final.T1SG2_8522605005567,Batch_2,2026-05-07,2026-05-07,T1SGL1,NA,Sequenced
T1SG3,T1,seagrass,prokaryote,zostera_capensis,zostera_capensis,Real Sample,RemovePrimer_Final.T1SG3_8522605005568,Batch_2,2026-05-07,2026-05-07,T1SGL1,NA,Sequenced
sample,transect,habitat,microbial_community,plant_species_1,plant_species_2,blank_group,Read_tag,batch,date,assigned_date,lab_blank,field_blank,sequenced_tag


In [38]:
example <- cbind.data.frame(c("OTU1","OTU2","OTU3","OTU4","OTU5","OTU6"),
                        c(0,200,1000,50,0,25),
                        c(0,220,800,30,0,10),
                        c(0,180,1300,70,0,30),
                        c(60,660,1440,70,2400,30),
                        c(64,520,1000,48,1900,20),
                        c(40,480,700,35,2100,15),
                        c("K_Bacteria; P_Actinobacteria","K_Bacteria; P_Proteobacteria","K_Bacteria; P_Proteobacteria","K_Bacteria; P_Bacteroidetes","K_Bacteria","K_Bacteria"))
colnames(example) <- c("OTU_ID","Blank1","Blank2","Blank3","Pop1_Sample1","Pop1_Sample2","Pop2_Sample3","Taxa")

In [20]:
example

OTU_ID,Blank1,Blank2,Blank3,Pop1_Sample1,Pop1_Sample2,Pop2_Sample3,Taxa
<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>
OTU1,0,0,0,60,64,40,K_Bacteria; P_Actinobacteria
OTU2,200,220,180,660,520,480,K_Bacteria; P_Proteobacteria
OTU3,1000,800,1300,1440,1000,700,K_Bacteria; P_Proteobacteria
OTU4,50,30,70,70,48,35,K_Bacteria; P_Bacteroidetes
OTU5,0,0,0,2400,1900,2100,K_Bacteria
OTU6,25,10,30,30,20,15,K_Bacteria


In [21]:
phyloseq::otu_table(phyloseq_16S) |>
data.frame() |>
tibble::rownames_to_column(var = "OTU_ID") |>  head(3)

,OTU_ID,RemovePrimer_Final.T1IS1_8522605005578,RemovePrimer_Final.T1IS2_8522605005579,RemovePrimer_Final.T1IS3_8522605005580,RemovePrimer_Final.T1ISL1_8522605005601,RemovePrimer_Final.T1L1_8522605005596,RemovePrimer_Final.T1SG1_8522605005566,RemovePrimer_Final.T1SG2_8522605005567,RemovePrimer_Final.T1SG3_8522605005568,RemovePrimer_Final.T1SGB_8522605005569,⋯,RemovePrimer_Final.T3SGL3_8522605005600,RemovePrimer_Final.T3SS1_8522605005593,RemovePrimer_Final.T3SS2_8522605005594,RemovePrimer_Final.T3SS3_8522605005595,RemovePrimer_Final.T3SSL3_8522605005605,RemovePrimer_Final.T3W1_8522605005562,RemovePrimer_Final.T3W2_8522605005563,RemovePrimer_Final.T3W3_8522605005564,RemovePrimer_Final.T3WB_8522605005565,RemovePrimer_Final.TSGBL1_8522605005606
,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1,8b41a43af1ba975e1a4b475851c266d0,0,0,0,0,0,0,3,2,0,⋯,0,0,0,0,0,0,0,0,0,0
2,75b76ef263e9257e5510adbc49acfc05,0,0,0,0,0,0,0,0,0,⋯,0,0,0,0,0,0,0,0,0,0
3,20d6ed2f2e5a2e80300fe1bd1d8b035d,0,0,0,0,0,0,0,0,0,⋯,0,0,0,0,0,0,0,0,0,0


In [22]:
?microDecon::decon

decon {microDecon},R Documentation
data,"A data frame of metabarcoding read data consisting of at least 3 columns in this order: a column of unique OTU names/labels, at least one column of read data from a blank sample (this contains your known contaminant reads), at least one column of read data for an actual sample (each column is a sample, each row is an OTU, and each cell is the number of reads). It can optionally include a final column with taxonomy information. If multiple blanks are included (recommended), they must be in consecutive columns, starting with column 2. Individuals must be ordered by group (e.g., species, populations, etc.)."
numb.blanks,"Numeric (default = 1). Specifies the number of blanks included in the data set (if multiple blanks are included, they must be in consecutive columns, starting with column 2)."
numb.ind,"A vector of numbers listing the number of individuals in each user-specified group (e.g., different populations or different species could be treated as different groups). Data must be sorted by these groups beforehand."
taxa,Logical (T/F). Specifies whether or not the last column contains taxonomic information (default = T)
runs,"Numeric (default = 2). Specifies the number of times that the function should run the decontamination procedure on the data. Based on simulation results, using two runs is best on average, but using one run is better if there is very little contamination, and using more than two runs is better if there is substantial contamination (see User Guide section 1.4.3)."
thresh,"Numeric (default = 0.7). A number written as a proportion. This is the threshold at which if that proportion of 0s are present for an OTU within a group, all samples will be set to 0 for that OTU for that group (e.g., if thresh = 0.7, then if, for a particular OTU, 70 percent of samples are 0 within a group, all samples become 0 for that OTU). The threshold always rounds down to calculate the maximum number of zeros that can be present (e.g., if thresh = 0.7 and there are 11 samples, then any OTU with 7 or more 0s will become 0 for all samples in that group). It will not do anything to groups with four or fewer samples. Set to 1 if you do not want to apply this threshold"
prop.thresh,"Numeric (default = 0.00005). A number written as a proportion. This is the threshold at which if the number of reads for a particular OTU are below this proportion, the OTU will be set to zero for all individuals in that group (e.g., if a particular OTU makes up 0.001 percent of all of the reads for a group, then at prop.thresh = 0.00005, that OTU would be set to 0 for all individuals in the group [0.00005 = 0.005 percent]). The proportions are based on all reads for all individuals in a group (including OTUs that were not in the blank). It is necessary to relax this threshold (e.g., 0.0005) for very small data sets (see User Guide section 1.4.4) Set to 0 if you do not want to use this threshold."
regression,Numeric (default = 0). Specifies the regression equation used to calculate the constant. 0 = it chooses between regression 1 and regression 2 based on the low.threshold and up.threshold arguments (this is strongly recommended). 1 = it always uses regression 1. 2 = it always uses regression 2. See User Guide section 1.4.2.
low.threshold,Numeric (default = 40). Selects the lower point for switching between regression 1 and regression 2. It uses regression 2 anytime that the estimated overlap is <low.threshold or >up.threshold. It is usually best not to change this value.
up.threshold,Numeric (default = 400). Selects the higher point for switching between regression 1 and regression 2. It uses regression 2 anytime that the estimated overlap is <low.threshold or >up.threshold. It is usually best not to change this value.
